# Target Tracking (With Yaw Compensation)

This tutorial demonstrates how to generate a target tracking profile, aligning with the target velocity direction

## Setup

In [1]:
%pip install cesiumpy@git+https://github.com/open-space-collective/cesiumpy

In [2]:
import os
import cesiumpy

from ostk.mathematics.object import RealInterval

from ostk.physics import Environment
from ostk.physics.unit import Length
from ostk.physics.unit import Angle
from ostk.physics.time import Scale
from ostk.physics.time import Instant
from ostk.physics.time import Duration
from ostk.physics.time import Interval
from ostk.physics.time import DateTime
from ostk.physics.time import Time
from ostk.physics.coordinate import Position
from ostk.physics.coordinate.spherical import LLA

from ostk.astrodynamics import Trajectory
from ostk.astrodynamics.trajectory import Orbit
from ostk.astrodynamics.trajectory.model import TargetScan
from ostk.astrodynamics.flight import Profile
from ostk.astrodynamics.access import Generator as AccessGenerator
from ostk.astrodynamics.access import AccessTarget
from ostk.astrodynamics.access import VisibilityCriterion
from ostk.astrodynamics.viewer import Viewer
from ostk.astrodynamics.viewer import ConicSensor

---

## Computation

Initialize an environment:

In [3]:
environment = Environment.default(True)
earth = environment.get_celestial_object_with_name("Earth")

Construct an orbit:

In [4]:
orbit = Orbit.sun_synchronous(
    epoch=Instant.date_time(DateTime(2024, 1, 1, 0, 0, 0), Scale.UTC),
    altitude=Length.kilometers(550.0),
    local_time_at_descending_node=Time(14, 30, 0),
    celestial_object=earth,
)

Define an interval of interest:

In [5]:
interval = Interval.closed(
    start_instant=Instant.date_time(DateTime(2024, 2, 9, 0, 0, 0), Scale.UTC),
    end_instant=Instant.date_time(DateTime(2024, 2, 10, 0, 0, 0), Scale.UTC),
)

Define the coordinates of the target to track:

In [6]:
start_lla = LLA(
    latitude=Angle.degrees(0.0),
    longitude=Angle.degrees(0.0),
    altitude=Length.meters(0.0),
)
end_lla = LLA(
    latitude=Angle.degrees(0.5),
    longitude=Angle.degrees(0.1),
    altitude=Length.meters(0.0),
)
mid_lla = start_lla.calculate_intermediate_to(end_lla, 0.5)

Find one access during which the satellite is over the target:

In [7]:
visibility_criterion = VisibilityCriterion.from_aer_interval(
    azimuth_interval=RealInterval.closed(0.0, 360.0),
    elevation_interval=RealInterval.closed(60.0, 90.0),
    range_interval=RealInterval.closed(0.0, 10000e3),
)
access_target = AccessTarget.from_lla(
    visibility_criterion=visibility_criterion,
    lla=mid_lla,
    celestial=earth,
)
access_generator = AccessGenerator(
    environment=environment,
)

accesses = access_generator.compute_accesses(
    interval=interval,
    access_target=access_target,
    to_trajectory=orbit,
)

access = accesses[0]

In [8]:
time_step = Duration.seconds(1.0)
instants = access.get_interval().generate_grid(time_step)

Construct a Vehicle Velocity, Local Horizontal (VVLH) frame from the orbit:

In [9]:
vvlh_frame = orbit.get_orbital_frame(Orbit.FrameType.VVLH)

Generate a flight profile:

In [10]:
line_scan_trajectory = Trajectory(
    TargetScan(
        start_lla=start_lla,
        end_lla=end_lla,
        start_instant=access.get_acquisition_of_signal(),
        end_instant=access.get_loss_of_signal(),
        step_size=time_step,
        celestial=earth,
    )
)

In [11]:
uncompensated_profile = Profile.aligned_and_constrained_pointing(
    orbit=orbit,
    alignment_pointing_target=Profile.TrajectoryPointingTarget.target_position(
        trajectory=line_scan_trajectory,
        direction_in_body_frame=[0.0, 0.0, 1.0],
    ),
    clocking_pointing_target=Profile.PointingTarget(
        type=Profile.PointingTarget.Type.VelocityECI,
        direction_in_body_frame=[1.0, 0.0, 0.0],
    ),
)

In [12]:
yaw_compensated_profile = Profile.aligned_and_constrained_pointing(
    orbit=orbit,
    alignment_pointing_target=Profile.TrajectoryPointingTarget.target_position(
        trajectory=line_scan_trajectory,
        direction_in_body_frame=[0.0, 0.0, 1.0],
    ),
    clocking_pointing_target=Profile.TrajectoryPointingTarget.target_sliding_ground_velocity(
        trajectory=line_scan_trajectory,
        direction_in_body_frame=[1.0, 0.0, 0.0],
    ),
)

Visualize the scene in 3D:

In [13]:
# Get a token from https://cesium.com/learn/ion/cesium-ion-access-tokens/
viewer = Viewer(
    interval=access.get_interval(),
    cesium_token=os.environ.get("CESIUM_TOKEN"),
)

viewer.add_profile(
    profile=yaw_compensated_profile,
    step=Duration.seconds(10.0),
    show_orbital_track=True,
    cesium_asset_id=669199,
    sensors=[
        ConicSensor(
            "Boresight",
            [0.0, 0.0, 1.0],
            cesiumpy.color.RED.with_alpha(0.5),
            Angle.degrees(0.5),
            Length.meters(6913000.0),
        )
    ],
)

viewer.add_line(
    positions=[Position.from_lla(lla) for lla in LLA.linspace(start_lla, end_lla, 100)]
)

In [14]:
with open("render.html", "w") as f:
    f.write(viewer.render())

---